# 04 - Benchmark data access

Is the conversion worth it? Let's measure it under different **access patterns**:

| Dataset | Access patterns |
|---|---|
| DSM | small area (100 m x 100 m), full tile |
| RGB | small area (50 m x 50 m), thumbnail (512 x 512 px) |
| Temperature | map for a single day, time series at location |

We use [cloudgeometer](https://github.com/CLOUD-NES/cloudgeometer), a small tool we are building to facilitate these types of experiments. For each benchmark, it:

* repeats the read several times;
* runs a local HTTP proxy that logs **every request** made by the reader: number of requests, bytes transferred, and byte ranges.

Timings depend on network, load on the object store, caching, etc. Request counts and bytes are (mostly!) deterministic and help to explain timings.

In [ ]:
import datetime
import logging
import os
import pathlib
import statistics
import typing

import fsspec
import matplotlib.pyplot as plt
import pandas as pd
import rasterio
import xarray as xr
from cloudgeometer import Benchmark

from utils import ENDPOINT_URL, BUCKET_URI, USER_PREFIX_URI, gdal_env, icechunk_repo

In [ ]:
fs = fsspec.filesystem("s3", endpoint_url=ENDPOINT_URL)

## A first benchmark

Read a full DSM tile with `rasterio`:

In [ ]:
uri = f"{BUCKET_URI}/AHN-4/DSM05/AHN4_R_135000_454000.TIF"
bbox = (135400, 454400, 135600, 454600)  # (minx, miny, maxx, maxy) in EPSG:28992

def read_bbox(href, bbox=None):
    with gdal_env(), rasterio.open(href) as src:
        window = src.window(*bbox) if bbox else None
        return src.read(window=window)

benchmark = Benchmark(
    uri, 
    reader=read_bbox, 
    reader_params={"bbox": bbox},
    num_runs=3, 
    log_requests=True)
result = benchmark.run()

print(result.summarize(verbose=False))

> ### Challenge
> * Why the runs are not identical?
> * What is GDAL doing with the first request? (Hint: follow the URL)
> * How many bytes are downloaded to read 400 x 400 float32 values (640 kB)?
>   
> You can check the details of the requests of a single run as:
> ```python
> result.runs[0].request_logs.to_df()
> ```

Caching is great in practice, but it makes repeated runs incomparable. For the rest of the benchmarks we'll disable GDAL's HTTP cache, so that every run starts "cold". We also tell GDAL for only fetch the file we asked for:

In [ ]:
os.environ["CPL_VSIL_CURL_NON_CACHED"] = "/vsis3/"
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"

## Run all benchmarks

A helper function to run a benchmark and store some summary statistics:

In [ ]:
def bench(
    dataset: str, 
    variant: str, 
    pattern: str, 
    href: str, 
    reader: typing.Callable, 
    params: dict | None = None,
    num_runs: int = 3
) -> None:
    """Run a benchmark for a reader function."""
    benchmark = Benchmark(
        href, 
        reader, 
        params, 
        num_runs=num_runs, 
        log_requests=True
    )
    result = benchmark.run()
    ok = [run for run in result.runs if run.success]
    if not ok:
        raise RuntimeError(
            f"FAILED: {dataset}/{variant}/{pattern}: {result.runs[0].error}"
        )
    return {
        "user": USER_PREFIX_URI.rsplit("/", 1)[-1],
        "dataset": dataset,
        "variant": variant,
        "pattern": pattern,
        "time_s": statistics.median(run.time for run in ok),
        "requests": len(ok[0].request_logs),
        "size_MB": ok[0].request_logs.total_bytes / 1e6,
    }

### Digital Surface Model

Let's check the original tile against all the variants you have created in notebook 02

In [ ]:
DSM_ORIGINAL_URI = f"{BUCKET_URI}/AHN-4/DSM05/AHN4_R_136000_455000.TIF"
DSM_BBOX = (136400, 455400, 136600, 455600)

In [ ]:
dsm_hrefs = {"original": DSM_ORIGINAL_URI}
dsm_hrefs.update(**{
    pathlib.PurePath(path).stem: f"s3://{path}"
    for path in fs.glob(f"{USER_PREFIX_URI}/AHN-4/DSM05/variants/*.tif")
})
dsm_hrefs

In [ ]:
dsm_results = []
for variant, href in dsm_hrefs.items():
    dsm_results.extend([
        bench("DSM", variant, "small area", href, read_bbox, {"bbox": DSM_BBOX}),
        bench("DSM", variant, "full tile", href, read_bbox)
    ])

In [ ]:
dsm_results_df = pd.DataFrame(dsm_results)
table = dsm_results_df.pivot_table(
    index=["variant"], 
    columns="pattern", 
    values=["time_s", "requests", "size_MB"]
)
table

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)#)
for n, metric in enumerate(("time_s", "requests", "size_MB")):
    table[metric].plot.barh(ax=axes[n], title=f"{metric}", legend=metric == "time_s")

> ### Challenge
> Which DSM variant is fastest for the small area? And for the full tile? Explain it with the number of requests and bytes.


### RGB Aerial Images

In [ ]:
RGB_ORIGINAL_URI = f"{BUCKET_URI}/aerial-images-NL-2024/RGB/2024_136000_455000_RGB_hrl.tif"
RGB_BBOX = (136400, 454400, 136450, 454450)

def read_thumbnail(href, size=512):
    with gdal_env(), rasterio.open(href) as src:
        return src.read(out_shape=(src.count, size, size))  # GDAL picks the closest overview

rgb_hrefs = {"original": RGB_ORIGINAL_URI} 
rgb_hrefs.update(**{
    pathlib.PurePath(path).stem: f"s3://{path}"
    for path in fs.glob(f"{USER_PREFIX_URI}/aerial-images-NL-2024/RGB/variants/*.tif")
})
rgb_hrefs

In [ ]:
rgb_results = []
for variant, href in rgb_hrefs.items():
    rgb_results.extend([
        bench("RGB", variant, "small area", href, read_bbox, {"bbox": RGB_BBOX}),
        bench("RGB", variant, "thumbnail", href, read_thumbnail)
    ])

In [ ]:
rgb_results_df = pd.DataFrame(rgb_results)
table = rgb_results_df.pivot_table(
    index=["variant"], 
    columns="pattern", 
    values=["time_s", "requests", "size_MB"]
)
table 

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)#)
for n, metric in enumerate(("time_s", "requests", "size_MB")):
    table[metric].plot.barh(ax=axes[n], title=f"{metric}", legend=metric == "time_s")

> ### Challenge
> How much faster is the thumbnail with overviews?

### Gridded Daily Mean Temperature

The original NetCDF files can only be benchmarked for the single-day map: a time series would require downloading all 365 files. The virtual Icechunk store gives Zarr access to the *same bytes*.

In [ ]:
X, Y = 136000, 455000  # location for the time series (EPSG:28992)
DAY = "2025-07-01"


def select(ds: xr.Dataset, pattern: str) -> xr.DataArray:
    if pattern == "time-series":
        return ds["prediction"].sel(x=X, y=Y, method="nearest")
    elif pattern == "map":
        return ds["prediction"].sel(time=DAY)
    else:
        raise ValueError(f"Unknown pattern: {pattern}")


def read_netcdf(href, pattern):
    with fsspec.filesystem("s3", skip_instance_cache=True, endpoint_url=ENDPOINT_URL).open(href) as f, xr.open_dataset(f, engine="scipy") as ds:
        return select(ds, pattern).values


def read_zarr(href, pattern):
    ds = xr.open_zarr(href, consolidated=True, zarr_format=3, storage_options={"endpoint_url": ENDPOINT_URL, "skip_instance_cache": True})
    return select(ds, pattern).values


def read_icechunk(href, pattern):
    store = icechunk_repo(href).readonly_session("main").store
    ds = xr.open_zarr(store)
    return select(ds, pattern).values

In [ ]:
# fresh filesystem instances (skip_instance_cache), so that the request-logging proxy is picked up/
# These are never closed explicitly: silence the resulting "Unclosed client session" messages
logging.getLogger("asyncio").setLevel(logging.CRITICAL)


temp_results = []
netcdf_day = fs.glob(f"{BUCKET_URI}/KNMI-Tg1-5/{DAY.replace('-', '')}/*.nc")[0]
temp_results.append(
    bench("Temperature", "original", "map", f"s3://{netcdf_day}", read_netcdf, {"pattern": "map"})
)
for pattern in ("time-series", "map"):
    temp_results.append(
        bench("Temperature", "virtual", pattern, f"{USER_PREFIX_URI}/KNMI-Tg1-5/icechunk", read_icechunk, {"pattern": pattern})
    )
    for path in fs.glob(f"{USER_PREFIX_URI}/KNMI-Tg1-5/zarr/*.zarr"):
        layout = pathlib.PurePath(path).stem
        temp_results.append(
            bench("Temperature", layout, pattern, f"s3://{path}", read_zarr, {"pattern": pattern})
        )

In [ ]:
temp_results_df = pd.DataFrame(temp_results)
table = temp_results_df.pivot_table(
    index=["variant"], 
    columns="pattern", 
    values=["time_s", "requests", "size_MB"]
)
table

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)#)
for n, metric in enumerate(("time_s", "requests", "size_MB")):
    table[metric].plot.barh(ax=axes[n], title=f"{metric}", legend=metric == "time_s")

> ### Challenge
> * Which Zarr layout would you publish, given that users do not all access the data in the same way?
> * The single-day map from the virtual store reads the same bytes as the original file: is it as fast?

## Save the results and compare with the group

Everyone ran the same benchmarks, at the same time, on the same infrastructure. How reproducible are the timings?

In [ ]:
results = pd.concat([dsm_results_df, rgb_results_df, temp_results_df], ignore_index=True)

In [ ]:
fs.pipe(f"{USER_PREFIX_URI}/results/benchmark.json", results.to_json(orient="records").encode())
print(fs.ls(f"{USER_PREFIX_URI}/results"))

In [ ]:
all_results = pd.concat(
    [pd.read_json(fs.open(path)) for path in fs.glob(f"{BUCKET_URI}/users/*/results/benchmark.json")]
)
print("Participants:", all_results["user"].nunique())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 6), constrained_layout=True)
for ax, (dataset, df) in zip(axes, all_results.groupby("dataset")):
    df.boxplot(column="time_s", by=["variant", "pattern"], ax=ax, vert=False)
    ax.set_title(dataset)
    ax.set_xlabel("time (s)")
fig.suptitle("");

> ### Discussion
> * How large is the spread across participants, compared to the differences between variants?
> * All benchmarks ran concurrently on the same VM and object store: what does that mean for the conclusions?